# SPY 0DTE Credit Spreads — direction A/B + strike-aggression sweep

**Window:** last 365 calendar days (≈252 trading days).

**Setup:** Opening range = first 30 min. First 1-min close outside OR between 10:00 ET and 13:00 ET (10:00 AM PT) triggers a credit spread.

**Direction modes (the key A/B):**
- **`continuation`** — break above ORH → bull put (bet stays high); break below ORL → bear call (bet stays low). Previous run got 14% WR → losing.
- **`reversion`** — break above ORH → bear call (bet fades back down); break below ORL → bull put (bet bounces back up). Hypothesis: should get ~85% WR.

**Sweep:** 7 PT × 3 SL × 3 TS × 4 strike offsets × 2 directions = **504 configs**.

Cache is warmed for the continuation side; the new reversion side reuses the same option-bar data (just trades the opposite spread type). Should add only ~5 min.

## 1. Setup

In [ ]:
import os, sys
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
if not os.path.isdir(REPO):
    !git clone --quiet -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO}
else:
    !cd {REPO} && git pull --quiet
SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Smoke test

In [ ]:
!python -m iron_condor.cli --smoke -v

## 4. Full sweep — 504 configs (continuation + reversion A/B)

In [ ]:
!python -m iron_condor.cli --days 365 --sweep \
    --short-offset 0 --short-offset 1 --short-offset 2 --short-offset 3 \
    --direction continuation --direction reversion

## 5. Top 20 configs

In [ ]:
import pandas as pd
summary = pd.read_csv('results/sweep_summary.csv')
summary.head(20)

## 6. The A/B headline — best continuation vs best reversion

In [ ]:
for dir_short, label in [('cont', 'continuation'), ('reve', 'reversion')]:
    s = summary[summary['config'].str.contains(f'dir={dir_short}')]
    if not s.empty:
        best = s.iloc[0]
        print(f'\n=== Best {label.upper()} config ===')
        print(f"  config:       {best['config']}")
        print(f"  return:       {best['total_return_pct']:>+8.1f}%")
        print(f"  win rate:     {best['win_rate']:>8.1%}")
        print(f"  trades:       {best['n_trades']}")
        print(f"  avg net P&L:  ${best['avg_net_pnl']:>+8.2f}")
        print(f"  max DD:       {best['max_drawdown_pct']:>+8.1f}%")

## 7. PT × SL heatmaps, broken out by direction × strike offset

In [ ]:
import re
rows = []
for _, r in summary.iterrows():
    m = re.match(r'pt(\d+)\|sl(\d+)\|ts(\d+)\|so([\d.]+)\|dir=(\w+)', r['config'])
    if m:
        rows.append({
            'pt': int(m.group(1)),
            'sl': int(m.group(2)),
            'ts': int(m.group(3)),
            'so': float(m.group(4)),
            'dir': m.group(5),
            'return': r['total_return_pct'],
            'win_rate': r['win_rate'],
        })
grid = pd.DataFrame(rows)
for d in ['reve', 'cont']:
    if d not in grid['dir'].values:
        continue
    print(f'\n#################  DIRECTION = {d.upper()}  #################')
    for off in sorted(grid['so'].unique()):
        g = grid[(grid['so'] == off) & (grid['dir'] == d)]
        if g.empty:
            continue
        print(f'\n=== short-offset = {off} ===')
        print('Return % (avg over ts):')
        print(g.pivot_table(index='pt', columns='sl', values='return', aggfunc='mean').round(1))
        print('Win rate (avg over ts):')
        print(g.pivot_table(index='pt', columns='sl', values='win_rate', aggfunc='mean').round(2))

## 8. Top config — breakdown by direction & exit reason

In [ ]:
trades = pd.read_csv('results/sweep_trades.csv')
top_cfg = summary.iloc[0]['config']
sub = trades[trades['config'] == top_cfg]
taken = sub[sub['exit_reason'].isin(['profit', 'stop', 'time_stop'])]
print(f'Top config: {top_cfg}')
print(f'\nTotal days: {len(sub)}, taken trades: {len(taken)}')
print('\n=== By direction ===')
print(taken.groupby('signal_direction').agg(
    trades=('net_pnl', 'count'),
    win_rate=('net_pnl', lambda s: (s > 0).mean()),
    avg_pnl=('net_pnl', 'mean'),
    total_pnl=('net_pnl', 'sum'),
).round(2))
print('\n=== Exit reasons ===')
print(sub['exit_reason'].value_counts())

## 9. Equity curves — best of each direction

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 5))
for d, label in [('reve', 'best reversion'), ('cont', 'best continuation')]:
    s = summary[summary['config'].str.contains(f'dir={d}')]
    if s.empty:
        continue
    cfg = s.iloc[0]['config']
    sub = trades[trades['config'] == cfg].sort_values('day')
    ax.plot(pd.to_datetime(sub['day']), sub['balance_after'], label=f'{label}: {cfg}', linewidth=1.5)
ax.axhline(1500, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date'); ax.set_ylabel('Balance ($)')
ax.set_title('Best of each direction — equity curves')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()